# Milestone 1 - SurfaceGuard AI (Industrial Surface Defect Segmentation)
Run the cells **top to bottom**. Every question prints a line starting with `ANSWER Qn` - copy those into the Google Form.

* Q1-Q5, Q7-Q11 only need `train.csv` (fast, no GPU).
* Q6 needs the images (about 1-2 minutes).
* Q12 trains a small U-Net (needs GPU: Settings -> Accelerator -> GPU P100/T4).

## 0. Setup

In [ ]:
import os, glob, time, warnings
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

H, W = 128, 800                     # native image size (height x width)
CANVAS = H * W                      # 102,400 pixels
CLASSES = ["defect_a", "defect_b", "defect_c", "defect_d"]

# ---- find train.csv automatically (works on Kaggle) ----
CSV = None
for base in ["/kaggle/input", "."]:
    for root, dirs, files in os.walk(base):
        if "train.csv" in files:
            CSV = os.path.join(root, "train.csv"); break
    if CSV: break
assert CSV, "train.csv not found - add the competition data via 'Add Input' on the right panel"
DATA_DIR = os.path.dirname(CSV)
print("train.csv  :", CSV)

df = pd.read_csv(CSV, keep_default_na=False)          # keep_default_na so empty RLE stays ''
df["mask_rle"] = df["mask_rle"].astype(str).str.strip()
df["pos"] = df["mask_rle"].ne("")                     # True = defect present
print(df.shape, "| unique images:", df.image_id.nunique())
df.head()

## Helper functions: RLE decode / encode (1-based, column-major / Fortran order)

In [ ]:
def rle_decode(rle, shape=(H, W)):
    """RLE string -> binary mask (H, W). 1-indexed, column-major."""
    flat = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    if isinstance(rle, str) and rle.strip():
        s = list(map(int, rle.split()))
        for start, length in zip(s[0::2], s[1::2]):
            flat[start - 1: start - 1 + length] = 1
    return flat.reshape(shape, order="F")

def rle_encode(mask):
    """binary mask (H, W) -> RLE string. 1-indexed, column-major. Empty mask -> ''."""
    pixels = np.concatenate([[0], mask.flatten(order="F").astype(np.uint8), [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return " ".join(map(str, runs))

# sanity check: decode -> encode must give the same string back
r = df.loc[df.pos, "mask_rle"].iloc[0]
assert rle_encode(rle_decode(r)) == r
print("RLE decode/encode round-trip OK")

## Q1 - Class distribution & clean images

In [ ]:
pos_counts = df.groupby("defect_type")["pos"].sum()
print(pos_counts.to_dict())
per_image = df.groupby("image_id")["pos"].sum()           # number of defect classes per image
n_clean = int((per_image == 0).sum())
print("completely clean images:", n_clean)
print(f"ANSWER Q1: {int(pos_counts.max()) + n_clean}   (most frequent = {pos_counts.idxmax()} {int(pos_counts.max())} + clean {n_clean})")

## Q2 - Multi-label co-occurrence

In [ ]:
print(per_image.value_counts().sort_index().to_dict(), "<- {labels per image: #images}")
n_multi = int((per_image > 1).sum())
print("ANSWER Q2:", n_multi)

## Q3 - Do any pixels carry 2+ defect labels? (spatial overlap)

In [ ]:
multi_ids = per_image[per_image > 1].index
sub = df[df.image_id.isin(multi_ids) & df.pos]
overlap_px = 0
for iid, g in sub.groupby("image_id"):
    stack = sum(rle_decode(r).astype(np.int32) for r in g.mask_rle)
    overlap_px += int((stack >= 2).sum())
print("images checked:", len(multi_ids))
print("ANSWER Q3:", overlap_px)

## Q4 - RLE boundary check

In [ ]:
bad_rows, max_idx = 0, 0
for r in df.loc[df.pos, "mask_rle"]:
    s = list(map(int, r.split()))
    ends = [a + l - 1 for a, l in zip(s[0::2], s[1::2])]
    max_idx = max(max_idx, max(ends))
    if max(ends) > CANVAS:
        bad_rows += 1
print(f"ANSWER Q4: {bad_rows} rows, max index {max_idx}")

## Q5 - Aspect-ratio distortion (128x800 -> 256x256)

In [ ]:
s_x = 256 / 800     # width 800 -> 256  (compression)
s_y = 256 / 128     # height 128 -> 256 (stretching)
print(f"ANSWER Q5: {s_x}, {s_y}, {s_y / s_x}")

## Q6 - Cohen's d (defect pixels vs local background, same image)
Needs the images. Pools all defect pixels and all background pixels of each class.
`d = (mean_defect - mean_bg) / pooled_std`.  We also measure *local texture* (local std-dev in a 7x7 window) to decide *intensity vs texture*.

In [ ]:
# ---- map image_id -> file path ----
img_paths = {}
for root, _, files in os.walk(DATA_DIR):
    for f in files:
        if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")):
            img_paths[os.path.splitext(f)[0]] = os.path.join(root, f)
print("images found:", len(img_paths))
missing = set(df.image_id.unique()) - set(img_paths)
print("train images missing:", len(missing))
assert not missing, "Add the competition data (with images) as notebook input"

def load_gray(iid):
    im = cv2.imread(img_paths[iid], cv2.IMREAD_GRAYSCALE)
    if im.shape != (H, W):
        print("WARNING shape", iid, im.shape); im = cv2.resize(im, (W, H))
    return im.astype(np.float64)

def local_std(im, k=7):
    m = cv2.blur(im, (k, k)); m2 = cv2.blur(im * im, (k, k))
    return np.sqrt(np.maximum(m2 - m * m, 0))

rows = []
for c in CLASSES:
    ids = df[(df.defect_type == c) & df.pos]
    nd = sd = sd2 = nb = sb = sb2 = 0.0
    tex_d = tex_b = 0.0
    for iid, r in zip(ids.image_id, ids.mask_rle):
        im = load_gray(iid); m = rle_decode(r).astype(bool)
        d, b = im[m], im[~m]
        if len(d) == 0 or len(b) == 0: continue
        nd += len(d); sd += d.sum(); sd2 += (d * d).sum()
        nb += len(b); sb += b.sum(); sb2 += (b * b).sum()
        ls = local_std(im); tex_d += ls[m].sum(); tex_b += ls[~m].sum()
    mu_d, mu_b = sd / nd, sb / nb
    var_d, var_b = sd2 / nd - mu_d ** 2, sb2 / nb - mu_b ** 2
    pooled = np.sqrt(((nd - 1) * var_d + (nb - 1) * var_b) / (nd + nb - 2))
    rows.append(dict(defect=c, mean_defect=mu_d, mean_bg=mu_b, pooled_std=pooled,
                     cohens_d=(mu_d - mu_b) / pooled,
                     local_texture_ratio=(tex_d / nd) / (tex_b / nb)))
cd = pd.DataFrame(rows).round(4)
display(cd)
inv = cd[cd.cohens_d.abs() <= 0.02]
print("classes with |d| <= 0.02:", inv.defect.tolist())
print("(local_texture_ratio > 1 means the defect is rougher/more textured than the background)")
print("ANSWER Q6 -> use the class listed above, feature type = texture (if its local_texture_ratio differs from 1)")

## Q7 - Naive Dice on an empty ground truth

In [ ]:
def dice_naive(pred, gt):
    p, g = int(pred), int(gt)         # sizes (|X|, |Y|) with inter = 0 here
    if p == 0 and g == 0: return 1.0
    return 0.0                         # 2*0/(p+g)
print(f"ANSWER Q7: {dice_naive(0, 0)}, {dice_naive(2, 0)}")

## Q8 - Official weighted metric on the toy batch

In [ ]:
fg = (6 * 0.80 + 4 * 0.40) / 10
fp = 1 - (4 * (1024 / 102400)) / 40
ea = 36 / 40
score = 0.70 * fg + 0.20 * fp + 0.10 * ea
print(f"ANSWER Q8: {fg:.4f}, {fp:.4f}, {ea:.4f}, {score:.4f}")

## Q9 - All-empty baseline on the full training set

In [ ]:
n_pairs = len(df); n_empty = int((~df.pos).sum())
naive = n_empty / n_pairs                      # Dice(0,0)=1 on empties, 0 on positives
official = 0.70 * 0.0 + 0.20 * 1.0 + 0.10 * 1.0   # no foreground credit; FP=1; EmptyAcc=1 on empties
print("empty pairs:", n_empty, "of", n_pairs)
print(f"ANSWER Q9: {naive:.4f}, {official:.4f}   (empty pairs = {n_empty})")

## Q10 - Connected components & minimum-area recall ceilings

In [ ]:
areas = {}
for c in CLASSES:
    a = []
    for r in df.loc[(df.defect_type == c) & df.pos, "mask_rle"]:
        n, lab, st, _ = cv2.connectedComponentsWithStats(rle_decode(r), connectivity=8)
        a += list(st[1:, cv2.CC_STAT_AREA])
    areas[c] = np.array(a)
    print(c, "components:", len(a), "| min area:", a and int(min(a)), "| total px:", int(np.sum(a)))

def lost_pct(c, N):
    a = areas[c]; return 100 * a[a < N].sum() / a.sum()

print(f"ANSWER Q10: {int(areas['defect_b'].min())}, {lost_pct('defect_b', 50):.2f}%, {lost_pct('defect_a', 200):.2f}%")

## Q11 - Multi-label combination stratified 80/20 split

In [ ]:
from sklearn.model_selection import train_test_split
P = df.pivot(index="image_id", columns="defect_type", values="pos")[CLASSES]
combo = P.apply(lambda r: "+".join(c for c in CLASSES if r[c]) or "clean", axis=1)
print(combo.value_counts().to_string())
n_combos = combo.nunique()

b_by_combo = combo[P["defect_b"]].value_counts()
expected_b_val = 0.2 * b_by_combo.sum()                # exact 80/20 in every stratum
print(f"ANSWER Q11: {n_combos}, {round(expected_b_val)}   (exact fractional = {expected_b_val:.1f})")

# the actual split we reuse for Q12 (and the rest of the project)
SEED = 42
train_ids, val_ids = train_test_split(combo.index.values, test_size=0.2,
                                      stratify=combo.values, random_state=SEED)
print("train/val images:", len(train_ids), len(val_ids),
      "| defect_b in val (real split):", int(P.loc[val_ids, "defect_b"].sum()))

## Q12 - Baseline model + threshold sweep
Small **U-Net written from scratch** (this will count as your "model built from scratch" later), trained at native 128x800 resolution.
Logs to Weights & Biases.

**Before running:** (1) turn GPU on, (2) add your W&B key as a Kaggle Secret named `WANDB_API_KEY` (Add-ons -> Secrets), (3) change `WANDB_PROJECT` below to your roll-number project name.

In [ ]:
# ===================== CONFIG =====================
EPOCHS        = 6          # 5-10 allowed by the question
BATCH_SIZE    = 16
LR            = 1e-3
SUBSET        = None       # e.g. 600 to smoke-test quickly; None = all data
USE_WANDB     = True
WANDB_PROJECT = "YOURROLLNO-t32026"      # <-- CHANGE THIS (e.g. 23f3001567-t32026)
RUN_NAME      = "m1-unet-scratch-baseline"
THRESHOLDS    = [0.3, 0.5, 0.7, 0.9]
# ==================================================
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

wb = None
if USE_WANDB:
    try:
        import wandb
        try:
            from kaggle_secrets import UserSecretsClient
            wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
        except Exception as e:
            print("Kaggle secret not used:", e); wandb.login()
        wb = wandb.init(project=WANDB_PROJECT, name=RUN_NAME,
                        config=dict(epochs=EPOCHS, batch=BATCH_SIZE, lr=LR, model="UNet-scratch",
                                    resolution=f"{H}x{W}", split_seed=SEED))
    except Exception as e:
        print("W&B disabled:", e); wb = None

In [ ]:
rle_map = {(i, d): r for i, d, r in zip(df.image_id, df.defect_type, df.mask_rle)}

class StripDataset(Dataset):
    def __init__(self, ids, aug=False):
        self.ids, self.aug = list(ids), aug
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        iid = self.ids[i]
        img = cv2.imread(img_paths[iid], cv2.IMREAD_GRAYSCALE)
        if img.shape != (H, W): img = cv2.resize(img, (W, H))
        msk = np.stack([rle_decode(rle_map[(iid, c)]) for c in CLASSES])      # (4, H, W)
        if self.aug:
            if np.random.rand() < 0.5: img, msk = img[:, ::-1], msk[:, :, ::-1]   # horizontal flip
            if np.random.rand() < 0.5: img, msk = img[::-1], msk[:, ::-1]         # vertical flip
        x = (img.astype(np.float32) / 255.0 - 0.5) / 0.25
        return (torch.from_numpy(np.ascontiguousarray(x))[None],
                torch.from_numpy(np.ascontiguousarray(msk).astype(np.float32)))

tr_ids = train_ids[:SUBSET] if SUBSET else train_ids
va_ids = val_ids[: SUBSET // 4] if SUBSET else val_ids
train_dl = DataLoader(StripDataset(tr_ids, aug=True), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(StripDataset(va_ids),           batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print("train batches:", len(train_dl), "| val batches:", len(val_dl))

def block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                         nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))

class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=4, f=(32, 64, 128, 256, 512)):
        super().__init__()
        self.enc = nn.ModuleList(); c = in_ch
        for o in f[:-1]: self.enc.append(block(c, o)); c = o
        self.pool = nn.MaxPool2d(2)
        self.mid = block(f[-2], f[-1])
        self.up = nn.ModuleList(); self.dec = nn.ModuleList(); c = f[-1]
        for o in reversed(f[:-1]):
            self.up.append(nn.ConvTranspose2d(c, o, 2, stride=2)); self.dec.append(block(o * 2, o)); c = o
        self.head = nn.Conv2d(f[0], out_ch, 1)      # 4 independent sigmoid outputs (multi-label)
    def forward(self, x):
        skips = []
        for e in self.enc: x = e(x); skips.append(x); x = self.pool(x)
        x = self.mid(x)
        for up, dec, s in zip(self.up, self.dec, reversed(skips)):
            x = dec(torch.cat([up(x), s], 1))
        return self.head(x)                          # raw logits

model = UNet().to(device)
print("params (M):", round(sum(p.numel() for p in model.parameters()) / 1e6, 2))

def dice_loss(logits, y, eps=1.0):
    p = torch.sigmoid(logits)
    inter = (p * y).sum((2, 3)); den = p.sum((2, 3)) + y.sum((2, 3))
    return 1 - ((2 * inter + eps) / (den + eps)).mean()

def loss_fn(logits, y):
    return F.binary_cross_entropy_with_logits(logits, y) + dice_loss(logits, y)

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=LR)
use_amp = device == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

for ep in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); tl = 0.0
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", enabled=use_amp):
            loss = loss_fn(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tl += loss.item() * len(x)
    tl /= len(train_dl.dataset)

    model.eval(); vl = 0.0
    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type="cuda", enabled=use_amp):
                vl += loss_fn(model(x), y).item() * len(x)
    vl /= len(val_dl.dataset)
    print(f"epoch {ep}/{EPOCHS}  train_loss {tl:.4f}  val_loss {vl:.4f}  ({time.time()-t0:.0f}s)")
    if wb: wb.log({"epoch": ep, "train_loss": tl, "val_loss": vl})

torch.save(model.state_dict(), "unet_scratch_m1.pt")

In [ ]:
@torch.no_grad()
def sweep(model, loader, thresholds):
    """Stream through the holdout; store per-(image,class) inter / pred-size / gt-size for each threshold."""
    model.eval()
    acc = {t: dict(i=[], p=[], g=[]) for t in thresholds}
    for x, y in loader:
        x, y = x.to(device), y.to(device).bool()
        with torch.autocast(device_type="cuda", enabled=use_amp):
            prob = torch.sigmoid(model(x).float())
        for t in thresholds:
            pr = prob > t
            acc[t]["i"].append((pr & y).sum((2, 3)).reshape(-1).cpu())
            acc[t]["p"].append(pr.sum((2, 3)).reshape(-1).cpu())
            acc[t]["g"].append(y.sum((2, 3)).reshape(-1).cpu())
    out = []
    for t in thresholds:
        i, p, g = [torch.cat(acc[t][k]).numpy().astype(np.float64) for k in "ipg"]
        dice = np.where((p + g) == 0, 1.0, 2 * i / np.maximum(p + g, 1))
        posm, emp = g > 0, g == 0
        fgd = dice[posm].mean() if posm.any() else 0.0
        fps = 1 - (p[emp] / CANVAS).mean() if emp.any() else 1.0
        eacc = (p[emp] == 0).mean() if emp.any() else 1.0
        out.append(dict(tau=t, naive_mean_dice=dice.mean(), fg_dice=fgd,
                        fp_pixel_rate_on_empty=(p[emp] / CANVAS).mean(),
                        pct_empty_pairs_with_any_fp=100 * (p[emp] > 0).mean(),
                        fp_score=fps, empty_acc=eacc,
                        official=0.7 * fgd + 0.2 * fps + 0.1 * eacc))
    return pd.DataFrame(out), float(emp.mean())

res, empty_frac = sweep(model, val_dl, THRESHOLDS)
res = res.round(4)
display(res)
print(f"Holdout empty-pair fraction (Q9-style count): {empty_frac:.4f}")
print("Peak tau - naive mean Dice:", res.loc[res.naive_mean_dice.idxmax(), "tau"],
      "| official score:", res.loc[res.official.idxmax(), "tau"])
if wb:
    wb.log({"threshold_sweep": wandb.Table(dataframe=res), "holdout_empty_fraction": empty_frac})
    wb.finish()

ax = res.plot(x="tau", y=["naive_mean_dice", "official", "fg_dice", "fp_score"], marker="o", figsize=(7, 4))
ax.set_title("Threshold sweep on 20% holdout"); plt.show()

### How to write the Q12 answer
Use your own table above, but the pattern to describe is:
* **Trend:** as tau goes 0.3 -> 0.9, false positives on empty pairs fall (fewer pixels, more pairs become exactly empty) while Foreground Dice on positive pairs falls (masks shrink, recall drops).
* **Which metric prefers higher tau:** naive mean Dice - about 84% of pairs are empty (Q9 count method), and each correctly-empty pair scores a perfect 1.0, while a single stray pixel scores 0.0. The official score gives 70% to foreground Dice, so it peaks at a lower/moderate tau where recall is not sacrificed.
* State the **peak tau** for each metric from your table and the holdout empty fraction.